In [24]:
import asyncio
import json
import os
import pandas as pd
from openai import AsyncOpenAI
from tenacity import retry, wait_exponential, stop_after_attempt
from tqdm.asyncio import tqdm

# ==========================================
# 1. 核心配置区
# ==========================================
# 替换为你的大模型 API 服务地址。
# 如果是你本地用 vLLM/Ollama 部署的开源模型，通常是 "http://localhost:8000/v1"
API_BASE_URL = "https://api.your-llm-provider.com/v1" 
API_KEY = "your_api_key_here"
MODEL_NAME = "qwen2.5-32b-instruct" # 或 "gpt-4o", "llama-3-70b"

INPUT_CSV = "raw_multilingual_data.csv"   # 需要标注的原始数据
OUTPUT_CSV = "annotated_data.csv"         # 标注后的输出文件
CONCURRENCY_LIMIT = 50                    # 最大并发请求数 (根据你的 API 速率调整)

# 确保我们的 6 大合规维度完全对齐
EXPECTED_KEYS = [
    "Hate_Speech", "Targeted_Harassment", "NSFW_Sexual", 
    "Violence_Incitement", "Dangerous_Ideology", "Profanity_Slang"
]

# ==========================================
# 2. 客户端与系统提示词初始化
# ==========================================
client = AsyncOpenAI(api_key=API_KEY, base_url=API_BASE_URL)
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

SYSTEM_PROMPT = """你是一个专业的全球多语言内容安全审核引擎。
请分析用户输入的文本，并严格按照以下 6 个维度进行二元打分（命中为 1.0，未命中为 0.0）。
维度定义：
- Hate_Speech: 针对种族、宗教、国籍等群体的仇恨与歧视。
- Targeted_Harassment: 针对具体个人的定向网暴或恶意辱骂。
- NSFW_Sexual: 色情、性暗示或违背意愿的性骚扰。
- Violence_Incitement: 明确的物理伤害威胁或煽动暴力。
- Dangerous_Ideology: 支持恐怖主义或极端分裂主义。
- Profanity_Slang: 包含粗俗脏话或冒犯性俚语。

你必须且只能返回一个合法的 JSON 对象，绝对不要输出任何其他解释性文字或 Markdown 标记。
输出格式示例：
{"Hate_Speech": 1.0, "Targeted_Harassment": 0.0, "NSFW_Sexual": 0.0, "Violence_Incitement": 0.0, "Dangerous_Ideology": 0.0, "Profanity_Slang": 1.0}"""

# ==========================================
# 3. 异步请求与容错逻辑
# ==========================================
# 使用 tenacity 实现指数退避重试：最多重试 5 次，间隔时间从 2 秒指数增加到 20 秒
@retry(wait=wait_exponential(multiplier=1, min=2, max=20), stop=stop_after_attempt(5))
async def annotate_single_text(text: str, index: int) -> dict:
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"输入文本：{text}"}
                ],
                temperature=0.0, # 必须设为 0，保证分类任务的确定性和一致性
                response_format={"type": "json_object"} # 强制 JSON 模式
            )
            
            raw_result = response.choices[0].message.content
            parsed_json = json.loads(raw_result)
            
            # 校验大模型是否漏掉了某些标签
            if not all(key in parsed_json for key in EXPECTED_KEYS):
                 raise ValueError(f"JSON 字段不完整: {parsed_json}")
                 
            return {"index": index, "labels": parsed_json, "status": "success"}
            
        except Exception as e:
            # 抛出异常以触发 @retry
            raise RuntimeError(f"Index {index} 标注失败: {str(e)}")

# ==========================================
# 4. 主控流与断点续传机制
# ==========================================
async def main():
    print("🚀 启动自动化多语言数据标注流水线...")
    
    # 1. 加载原始数据
    if not os.path.exists(INPUT_CSV):
        # 如果没有测试文件，我们动态生成几条用于测试
        print("未找到输入文件，正在生成跨国测试用例...")
        df = pd.DataFrame({
            "text": [
                "I love this beautiful world!", # 正常
                "Maldita puta, te vamos a matar.", # 西语: 脏话+暴力威胁
                "Dasar anjing, orang Cina pulang sana!", # 印尼语: 粗口+种族仇恨
                "هؤلاء الكفار يجب أن يموتوا" # 阿拉伯语: 极端意识形态/仇恨
            ]
        })
        df.to_csv(INPUT_CSV, index=False)
    else:
        df = pd.read_csv(INPUT_CSV)

    # 2. 断点续传逻辑
    if os.path.exists(OUTPUT_CSV):
        out_df = pd.read_csv(OUTPUT_CSV)
        processed_indices = set(out_df['index'].tolist())
        print(f"📦 发现已有进度，跳过已处理的 {len(processed_indices)} 条数据...")
    else:
        out_df = pd.DataFrame(columns=['index', 'text'] + EXPECTED_KEYS + ['status'])
        processed_indices = set()

    # 3. 构建待处理任务队列
    tasks = []
    task_indices = []
    for index, row in df.iterrows():
        if index not in processed_indices:
            tasks.append(annotate_single_text(row['text'], index))
            task_indices.append(index)

    if not tasks:
        print("✅ 所有数据已标注完毕！")
        return

    print(f"⚙️ 开始并发处理 {len(tasks)} 条新数据 (最大并发: {CONCURRENCY_LIMIT})...")
    
    # 4. 执行并发并显示进度条
    # tqdm.gather 可以在终端漂亮地展示进度和预估剩余时间
    results = await tqdm.gather(*tasks, return_exceptions=True)

    # 5. 解析结果并增量保存
    new_rows = []
    for res in results:
        if isinstance(res, Exception):
            # 如果彻底失败（重试 5 次依然报错），记录错误状态，防止阻塞大盘
            # 这里需要从 Exception 对象中提取是哪一个 index 失败的比较麻烦
            # 在实际工程中，通常直接记录到 error.log 中
            print(f"❌ 严重错误: {res}")
            continue
            
        idx = res["index"]
        row_data = {
            "index": idx,
            "text": df.at[idx, 'text'],
            "status": res["status"]
        }
        # 将 6 个维度的分数打平写入字典
        for key in EXPECTED_KEYS:
            row_data[key] = float(res["labels"].get(key, 0.0))
            
        new_rows.append(row_data)

    # 追加写入 CSV
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        if not os.path.exists(OUTPUT_CSV):
            new_df.to_csv(OUTPUT_CSV, index=False)
        else:
            new_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
            
    print(f"🎉 本轮标注完成！共成功处理 {len(new_rows)} 条数据，已保存至 {OUTPUT_CSV}。")

if __name__ == "__main__":
    # 解决 Windows 环境下 asyncio 可能报错的问题
    if os.name == 'nt':
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [5]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-44ef59c0fe4e47e992fb4e4d985efe3f",
    base_url="https://api.deepseek.com/v1"
)

response = client.chat.completions.create(
    model="deepseek-reasoner",
    messages=[
        {"role": "system", "content": "你是一个乐于助人的助手。"},
        {"role": "user", "content": "请介绍一下你自己。"}
    ],
    stream=False
)

print(response.choices[0].message.content)

你好！我是DeepSeek，一个由深度求索公司创造的AI助手，很高兴认识你！😊

让我简单介绍一下自己：
- 我是一个纯文本AI模型，专注于理解和生成自然语言
- 支持文件上传功能，可以处理图像、txt、pdf、ppt、word、excel等多种格式的文件，从中读取文字信息进行分析
- 完全免费使用，没有任何收费计划
- 拥有128K的上下文长度，能记住较长的对话内容
- 支持联网搜索功能（需要用户在界面手动开启）
- 可以通过官方应用商店下载App使用

我的知识截止到2024年7月，会尽我所能热情细致地帮助你解答问题、处理任务。无论是学习、工作还是日常生活中的疑问，我都很乐意为你提供帮助！

有什么我可以为你做的吗？


In [ ]:
import os
import numpy as np
from transformers import AutoModel
from openai import OpenAI

# Load model directly from HF
model = AutoModel.from_pretrained(
    "govtech/lionguard-2", 
    trust_remote_code=True
    )

# Get OpenAI embeddings (users to input their own OpenAI API key)
client = OpenAI(api_key="sk-JXgZv4AzWNPDKuiXE9ixc0w9mBJdkJoGJz19SBjy6bn0aY6p")
response = client.embeddings.create(
    input="Hello, world!", # users to input their own text
    model="text-embedding-3-large",
    dimensions=3072 # dimensions of the embedding
    )
embeddings = np.array([data.embedding for data in response.data])

# Run LionGuard 2
results = model.predict(embeddings)


Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


AttributeError: 'FloatProgress' object has no attribute 'style'

In [ ]:
from openai import OpenAI

# 初始化客户端（替换成你的第三方API地址和key）
client = OpenAI(
    base_url="https://yinli.one/v1",
    api_key="sk-iWDUaAd5jiannRyNofybpNUZexIoEmx9y85ct27sDa2arn0a"
)

# 创建向量嵌入（Embedding）
response = client.embeddings.create(
    model="text-embedding-3-large",
    input="The food was delicious and the waiter...",
    encoding_format="float"
)

# 打印结果
print(response.data[0].embedding)

[-0.010992265, 0.0027546093, -0.0057185953, -0.008060995, 0.0006436691, 0.010560427, -0.011234357, -0.056165244, -0.010501539, 0.03946747, -0.0047600437, 0.009192936, -0.061766066, 0.07291536, -0.0010280713, -0.0033140373, -0.01228124, -0.012333584, -0.03643151, 0.003035959, -0.02189293, -0.015153623, 0.058363695, -0.016239764, 0.06134731, -0.024497049, -0.012516788, 0.024353104, 0.037426047, 0.015323741, -0.015572377, 0.016737033, 0.038917854, -0.004387092, -0.024575565, 0.02037495, 0.019301895, -0.011404475, 0.06537781, 0.032453354, -0.052291777, 0.003467798, 0.014682527, -0.01981225, -0.032715075, -0.057840254, 0.0075898976, 0.0053979876, -0.041979987, -0.009127506, 0.0129748, -0.015075107, 0.031720538, 0.013962795, -0.009984641, 0.01961596, -0.029495914, -0.014499322, -0.013321579, -0.032872107, -0.01000427, -0.03972919, 0.004910533, -0.008532092, 0.023083758, 0.022036875, 0.009487372, 0.05689806, -0.011208185, 0.006356539, 0.032034602, -0.015729409, -0.026564643, 0.004616097, 0.03

In [3]:
import pandas as pd

df = pd.read_parquet("./HateDay/hateday_v2_hf_final.parquet")
# print(df.head())
df

,tweet_id,text,class_clean,twitter_hate,violent_hate,target_majority,target_category,total_engagement,weighted,lang_country_hateday
0,1572562793207046148,@USER Sounds dumb coming off a loss,0,0,0,None,None,0,0,US
1,1572358197486166017,@USER @USER 🥱 LINK,0,0,0,None,None,3,0,US
2,1572337230013960192,"Okay, what are we getting at midnight tonight?...",0,0,0,None,None,5,0,US
3,1572373669355163651,If we get a performance of the lead single ton...,0,0,0,None,None,9,0,US
4,1572266825039351808,Mine are free LINK,0,0,0,None,None,1,0,US
...,...,...,...,...,...,...,...,...,...,...
539995,1572580249442951169,"#şirinyer İnsanca etkili en tesirlidir,Fyodor ...",0,0,0,None,None,47,1,tr
539996,1572321786628300803,ilam emir subayı kasnak basiretli dönüşmek bor...,0,0,0,None,None,38,1,tr
539997,1572559845592530945,paha Dünyadasın #kızılay #ankaratrv bir cok Ke...,0,0,0,None,None,56,1,tr
539998,1572298795362455552,uyuyacaksınMADDE3 de uykudurMADDE2 #corlu haki...,0,0,0,None,None,90,1,tr


In [8]:
print(df['lang_country_hateday'].unique())
print(len(df))
print(45000 *12)

['US' 'ar' 'de' 'en' 'es' 'fr' 'in' 'india' 'kenya' 'nigeria' 'pt' 'tr']
540000
540000


In [25]:
pt_list = df[(df['class_clean'] == 1) & (df['lang_country_hateday'] == 'pt')]
pt_list
# print(len(pt_list))
text_pt=pt_list['text']
# for t in list(text_pt):
#     print(t)

In [9]:
# import os
# # 创建输出文件夹（如果不存在）
# output_dir = "split_by_country"
# os.makedirs(output_dir, exist_ok=True)

# # 按 'lang_country_hateday' 列分组，并保存每个组
# for country, group in df.groupby('lang_country_hateday'):
#     # 构造文件名（如果国家名包含空格或特殊字符，可以替换）
#     safe_name = str(country).replace('/', '_')  # 处理可能的路径分隔符
#     filename = os.path.join(output_dir, f"{safe_name}.csv")
#     # 保存 CSV，不包含索引
#     group.to_csv(filename, index=False)
#     print(f"已保存 {country}: {len(group)} 条记录 -> {filename}")

In [ ]:
# 将多语言数据集汇总

Indonesia_path="./Indonesia"
Mexico_path="./Mexico"
Singapore_path="./Singapore"
Thailand_path="./Thailand"

Indonesia_path_file_1="/home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv"
Indonesia_path_file_2="/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt"
Indonesia_path_file_3="/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv"
Indonesia_path_file_4="/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv"


In [29]:
import pandas as pd
import os

# 文件路径（请根据实际情况调整）
Indonesia_path_file_1 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv"
Indonesia_path_file_2 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt"
Indonesia_path_file_3 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv"
Indonesia_path_file_4 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv"

# 用于存储所有推文的列表
all_tweets = []

# 文件1：格式 "Label,Tweet" (CSV，有表头)
print("正在处理文件1...")
df1 = pd.read_csv(Indonesia_path_file_1, encoding='utf-8')
if 'Tweet' in df1.columns:
    tweets1 = df1['Tweet'].dropna().tolist()
    all_tweets.extend(tweets1)
    print(f"文件1提取到 {len(tweets1)} 条推文")
else:
    print("文件1中未找到'Tweet'列，请检查文件格式")

# 文件2：格式 "Label\tTweet" (制表符分隔，无表头)
print("正在处理文件2...")
df2 = pd.read_csv(Indonesia_path_file_2, sep='\t', header=None, names=['Label', 'Tweet'], encoding='utf-8')
tweets2 = df2['Tweet'].dropna().tolist()
all_tweets.extend(tweets2)
print(f"文件2提取到 {len(tweets2)} 条推文")

# 文件3：格式 "Tweet,HS,Abusive,HS_Individual,..." (CSV，有表头，第一列为Tweet)
print("正在处理文件3...")
df3 = pd.read_csv(Indonesia_path_file_3, encoding='utf-8')
if 'Tweet' in df3.columns:
    tweets3 = df3['Tweet'].dropna().tolist()
    all_tweets.extend(tweets3)
    print(f"文件3提取到 {len(tweets3)} 条推文")
else:
    print("文件3中未找到'Tweet'列，请检查文件格式")

# 文件4：格式 "text,labels,source,dataset,nb_annotators" (CSV，第一列为text)
print("正在处理文件4...")
df4 = pd.read_csv(Indonesia_path_file_4, encoding='utf-8')
if 'text' in df4.columns:
    tweets4 = df4['text'].dropna().tolist()
    all_tweets.extend(tweets4)
    print(f"文件4提取到 {len(tweets4)} 条推文")
else:
    print("文件4中未找到'text'列，请检查文件格式")

# 汇总所有推文到一个DataFrame并保存
output_df = pd.DataFrame({'tweet': all_tweets})
output_file = "all_tweets_summary.csv"
output_df.to_csv(output_file, index=False, encoding='utf-8')
print(f"\n所有推文已汇总，共 {len(all_tweets)} 条，保存至 {output_file}")

# 可选：打印前几条查看
print("\n前5条推文示例：")
print(output_df.head())

正在处理文件1...
文件1提取到 2016 条推文
正在处理文件2...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x88 in position 10: invalid start byte

In [32]:
import pandas as pd
import chardet  # 可选，用于自动检测编码，如果没有安装可以注释掉

# 文件路径
Indonesia_path_file_1 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv"
Indonesia_path_file_2 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt"
Indonesia_path_file_3 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv"
Indonesia_path_file_4 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv"

def read_file_with_fallback(file_path, sep=',', header='infer', encoding_list=['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']):
    """尝试多种编码读取文件，返回DataFrame"""
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
    # 如果所有编码都失败，使用 latin-1 并忽略错误（保证能读，但可能乱码）
    print(f"  所有编码均失败，使用 latin-1 强制读取（可能含乱码）")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

all_tweets = []

# 文件1：CSV，列名 Label, Tweet
print("处理文件1...")
df1 = read_file_with_fallback(Indonesia_path_file_1, sep=',', header=0)
if 'Tweet' in df1.columns:
    tweets1 = df1['Tweet'].dropna().tolist()
    all_tweets.extend(tweets1)
    print(f"  提取 {len(tweets1)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df1.columns.tolist()}")

# 文件2：制表符分隔，第一行为表头
print("处理文件2...")
df2 = read_file_with_fallback(Indonesia_path_file_2, sep='\t', header=0)
if 'Tweet' in df2.columns:
    tweets2 = df2['Tweet'].dropna().tolist()
    all_tweets.extend(tweets2)
    print(f"  提取 {len(tweets2)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df2.columns.tolist()}")

# 文件3：CSV，第一列是 Tweet
print("处理文件3...")
df3 = read_file_with_fallback(Indonesia_path_file_3, sep=',', header=0)
if 'Tweet' in df3.columns:
    tweets3 = df3['Tweet'].dropna().tolist()
    all_tweets.extend(tweets3)
    print(f"  提取 {len(tweets3)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df3.columns.tolist()}")

# 文件4：CSV，推文列名为 text
print("处理文件4...")
df4 = read_file_with_fallback(Indonesia_path_file_4, sep=',', header=0)
if 'text' in df4.columns:
    tweets4 = df4['text'].dropna().tolist()
    all_tweets.extend(tweets4)
    print(f"  提取 {len(tweets4)} 条")
else:
    print(f"  未找到 'text' 列，实际列名: {df4.columns.tolist()}")

# 保存汇总结果
output_df = pd.DataFrame({'tweet': all_tweets})
output_file = "all_tweets_summary.csv"
output_df.to_csv(output_file, index=False, encoding='utf-8')
print(f"\n总计提取 {len(all_tweets)} 条推文，已保存至 {output_file}")
print("\n前5条预览：")
print(output_df.head())

处理文件1...
  使用编码 utf-8 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv
  提取 2016 条
处理文件2...
  使用编码 latin-1 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt
  提取 713 条
处理文件3...
  使用编码 latin-1 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv
  提取 13169 条
处理文件4...
  使用编码 utf-8 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv
  提取 14306 条

总计提取 30204 条推文，已保存至 all_tweets_summary.csv

前5条预览：
                                               tweet
0  - Dia sendiri yang ngiklanin promo cashback di...
1  - disaat semua cowok berusaha melacak perhatia...
2  - kampret kan kalo typo-nya di email kantor ke...
3  - Mending makan disini lebih murah, buang-buan...
4  /biarin oppa masukim vibrator ke memek/ oppa k...


In [33]:
import pandas as pd
import csv

# 文件路径
Indonesia_path_file_1 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv"
Indonesia_path_file_2 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt"
Indonesia_path_file_3 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv"
Indonesia_path_file_4 = "/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv"

def read_file_with_fallback(file_path, sep=',', header='infer', encoding_list=['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']):
    """尝试多种编码读取文件，返回DataFrame"""
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
    # 如果所有编码都失败，使用 latin-1 并忽略错误（保证能读，但可能乱码）
    print(f"  所有编码均失败，使用 latin-1 强制读取（可能含乱码）")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

all_tweets = []

# 文件1：CSV，列名 Label, Tweet
print("处理文件1...")
df1 = read_file_with_fallback(Indonesia_path_file_1, sep=',', header=0)
if 'Tweet' in df1.columns:
    tweets1 = df1['Tweet'].dropna().tolist()
    all_tweets.extend(tweets1)
    print(f"  提取 {len(tweets1)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df1.columns.tolist()}")

# 文件2：制表符分隔，第一行为表头
print("处理文件2...")
df2 = read_file_with_fallback(Indonesia_path_file_2, sep='\t', header=0)
if 'Tweet' in df2.columns:
    tweets2 = df2['Tweet'].dropna().tolist()
    all_tweets.extend(tweets2)
    print(f"  提取 {len(tweets2)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df2.columns.tolist()}")

# 文件3：CSV，第一列是 Tweet
print("处理文件3...")
df3 = read_file_with_fallback(Indonesia_path_file_3, sep=',', header=0)
if 'Tweet' in df3.columns:
    tweets3 = df3['Tweet'].dropna().tolist()
    all_tweets.extend(tweets3)
    print(f"  提取 {len(tweets3)} 条")
else:
    print(f"  未找到 'Tweet' 列，实际列名: {df3.columns.tolist()}")

# 文件4：CSV，推文列名为 text
print("处理文件4...")
df4 = read_file_with_fallback(Indonesia_path_file_4, sep=',', header=0)
if 'text' in df4.columns:
    tweets4 = df4['text'].dropna().tolist()
    all_tweets.extend(tweets4)
    print(f"  提取 {len(tweets4)} 条")
else:
    print(f"  未找到 'text' 列，实际列名: {df4.columns.tolist()}")

# 保存汇总结果，确保 CSV 格式正确（逗号、引号等自动转义）
output_df = pd.DataFrame({'tweet': all_tweets})
output_file = "all_tweets_summary.csv"
output_df.to_csv(output_file, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
print(f"\n总计提取 {len(all_tweets)} 条推文，已保存至 {output_file}")
print("CSV 已采用标准转义方式：包含逗号或引号的文本将被自动用双引号括起。")
print("\n前5条预览：")
print(output_df.head())

处理文件1...
  使用编码 utf-8 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv
  提取 2016 条
处理文件2...
  使用编码 latin-1 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt
  提取 713 条
处理文件3...
  使用编码 latin-1 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv
  提取 13169 条
处理文件4...
  使用编码 utf-8 成功读取 /home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv
  提取 14306 条

总计提取 30204 条推文，已保存至 all_tweets_summary.csv
CSV 已采用标准转义方式：包含逗号或引号的文本将被自动用双引号括起。

前5条预览：
                                               tweet
0  - Dia sendiri yang ngiklanin promo cashback di...
1  - disaat semua cowok berusaha melacak perhatia...
2  - kampret kan kalo typo-nya di email kantor ke...
3  - Mending makan disini lebih murah, buang-buan...
4  /biarin oppa masukim vibrator ke memek/ oppa k...


In [ ]:
import pandas as pd
import csv

# 定义文件配置列表
# 每个配置是一个字典，包含：
#   - path: 文件路径
#   - sep: 分隔符（如 ',' 或 '\t'），默认为 ','
#   - tweet_column: 推文所在的列名，默认为 'Tweet'（如果文件没有表头，需额外处理）
#   - header: 是否有表头，默认为 0（第一行是列名），若无表头则设为 None
#   - encoding_try: 可自定义尝试的编码顺序（可选）
file_configs = [
    {
        "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-abusive-language-detection/re_dataset_three_labels.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "header": 0
    },
    {
        "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt",
        "sep": "\t",
        "tweet_column": "Tweet",
        "header": 0
    },
    {
        "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "header": 0
    },
    {
        "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv",
        "sep": ",",
        "tweet_column": "text",
        "header": 0
    }
    # 您可以继续添加更多文件，例如：
    # {
    #     "path": "/path/to/another_file.csv",
    #     "sep": ",",
    #     "tweet_column": "content",
    #     "header": 0
    # }
]

def read_file_with_fallback(file_config):
    """
    根据配置尝试多种编码读取文件，返回DataFrame
    """
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)  # 默认第一行为列名
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    # 如果所有编码都失败，使用 latin-1 强制读取（忽略错误）
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取（可能含乱码）")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_tweets(df, tweet_column):
    """从DataFrame中提取推文列，去空，返回列表"""
    if tweet_column in df.columns:
        tweets = df[tweet_column].dropna().tolist()
        print(f"  提取到 {len(tweets)} 条推文")
        return tweets
    else:
        print(f"  ✗ 未找到列 '{tweet_column}'，实际列名: {df.columns.tolist()}")
        return []

def main():
    all_tweets = []
    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets = extract_tweets(df, config["tweet_column"])
        all_tweets.extend(tweets)
    
    # 汇总保存
    output_df = pd.DataFrame({'tweet': all_tweets})
    output_file = "all_tweets_summary.csv"
    output_df.to_csv(output_file, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 总计提取 {len(all_tweets)} 条推文，已保存至 {output_file}")
    print("CSV 已采用标准转义方式：包含逗号或引号的文本将被自动用双引号括起。")
    print("\n前5条预览：")
    print(output_df.head())

if __name__ == "__main__":
    main()

In [ ]:
import os
from openai import OpenAI

try:
    client = OpenAI(
        # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为: api_key="sk-xxx",
        api_key="sk-35805b6e284846f691c01e1b4caf4759",
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    completion = client.chat.completions.create(
        model="qwen-plus",  # 模型列表: https://help.aliyun.com/model-studio/getting-started/models
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user', 'content': '你是谁？具体的模型名称，是什么版本的'}
        ]
    )
    print(completion.choices[0].message.content)
except Exception as e:
    print(f"错误信息：{e}")
    print("请参考文档：https://help.aliyun.com/model-studio/developer-reference/error-code")

我是通义千问（Qwen），是阿里云研发的超大规模语言模型。我无法提供具体的版本号，因为我会持续进行更新和优化。如果您有具体的应用场景或问题需要帮助，我可以尽力为您提供支持。


In [18]:
import pandas as pd

def extract_csv_rows(input_file, output_file, start_row, end_row, header=True):
    """
    从 CSV 文件中提取指定行范围并保存到新文件。
    
    参数:
        input_file: 输入 CSV 文件路径
        output_file: 输出 CSV 文件路径
        start_row: 起始行号（从 1 开始计数，1 表示第一行数据）
        end_row: 结束行号（包含）
        header: 是否保留原文件的第一行作为表头（默认 True）
    """
    # 读取 CSV，如果 header=0 则第一行作为列名
    df = pd.read_csv(input_file, header=0 if header else None)
    
    # 行号转换：pandas 索引从 0 开始，且 header 占一行
    # 如果 header=True，第一行是列名，实际数据从索引 0 开始
    # start_row 和 end_row 是从数据行开始计数的（第 1 行数据对应索引 0）
    data_start_idx = start_row - 1
    data_end_idx = end_row - 1  # 包含结束行
    
    # 选择行范围
    selected = df.iloc[data_start_idx : data_end_idx + 1]
    
    # 保存到新文件，保留原列名
    selected.to_csv(output_file, index=False)
    print(f"已提取第 {start_row} 行到第 {end_row} 行，保存至 {output_file}")

# 使用示例
extract_csv_rows('new_annotated_data.csv', './Datasets/Annotated_LLM/new_en.csv', start_row=127801-7, end_row=127801+45000)

已提取第 127794 行到第 172801 行，保存至 ./Datasets/Annotated_LLM/new_en.csv


In [1]:
print(2016+713+13169+14306)

30204


In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_kaskus_search(keyword, pages=3):
    """
    通过关键词搜索抓取 Kaskus 论坛的帖子标题和片段
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    kaskus_data = []
    
    for page in range(1, pages + 1):
        print(f"正在抓取 Kaskus 第 {page} 页，关键词: {keyword}...")
        url = f"https://www.kaskus.co.id/search?q={keyword}&page={page}"
        
        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Kaskus 的搜索结果通常在特定的 div 类中，结构可能会随时间微调
            # 这里以提取帖子标题和摘要为例
            threads = soup.find_all("div", class_="thread-details") 
            
            for thread in threads:
                title_tag = thread.find("div", class_="thread-title")
                if title_tag:
                    title = title_tag.get_text(strip=True)
                    # 尝试获取内容摘要
                    snippet_tag = thread.find("div", class_="thread-content")
                    snippet = snippet_tag.get_text(strip=True) if snippet_tag else ""
                    
                    kaskus_data.append({
                        "platform": "Kaskus",
                        "text": f"{title} - {snippet}",
                        "author": "anonymous", # 搜索页通常不直接显示作者，如需可进一步请求帖子详情页
                        "url": "https://www.kaskus.co.id" + title_tag.find("a")["href"] if title_tag.find("a") else "",
                        "created_at": ""
                    })
            time.sleep(2)  # 礼貌性延时，防止被封 IP
        except Exception as e:
            print(f"抓取 Kaskus 第 {page} 页失败: {e}")
            
    df = pd.DataFrame(kaskus_data)
    print(f"成功抓取 {len(df)} 条 Kaskus 数据。")
    return df

# 使用示例 (抓取网络赌博相关黑话)
df_kaskus = scrape_kaskus_search("judi slot gacor", pages=5)
df_kaskus.to_csv("kaskus_judi.csv", index=False)

正在抓取 Kaskus 第 1 页，关键词: judi slot gacor...
抓取 Kaskus 第 1 页失败: HTTPSConnectionPool(host='www.kaskus.co.id', port=443): Max retries exceeded with url: /search?q=judi%20slot%20gacor&page=1 (Caused by NewConnectionError("HTTPSConnection(host='www.kaskus.co.id', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))
正在抓取 Kaskus 第 2 页，关键词: judi slot gacor...
抓取 Kaskus 第 2 页失败: HTTPSConnectionPool(host='www.kaskus.co.id', port=443): Max retries exceeded with url: /search?q=judi%20slot%20gacor&page=2 (Caused by NewConnectionError("HTTPSConnection(host='www.kaskus.co.id', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))
正在抓取 Kaskus 第 3 页，关键词: judi slot gacor...
抓取 Kaskus 第 3 页失败: HTTPSConnectionPool(host='www.kaskus.co.id', port=443): Max retries exceeded with url: /search?q=judi%20slot%20gacor&page=3 (Caused by NewConnectionError("HTTPSConnection(host='www.kaskus.co.id', port=443): Failed to establish a new connection: [Errno

KeyboardInterrupt: 

In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_kaskus_search(keyword, pages=3):
    """
    带有本地代理配置的 Kaskus 论坛抓取脚本
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # 【修改点 1】：在这里定义你的本地代理地址
    # 请务必将 7890 替换为你实际使用的科学上网工具的端口号（如 Clash 默认 7890，v2ray 默认 1080）
    proxies = {
        "http": "http://127.0.0.1:10808",
        "https": "http://127.0.0.1:10808"
    }
    
    kaskus_data = []
    
    for page in range(1, pages + 1):
        print(f"正在抓取 Kaskus 第 {page} 页，关键词: {keyword}...")
        url = f"https://www.kaskus.co.id/search?q={keyword}&page={page}"
        print(url)
        
        try:
            # 【修改点 2】：在发送网络请求时，将上面定义的 proxies 字典传进去
            response = requests.get(url, headers=headers, proxies=proxies, timeout=15)
            
            # 检查是否成功连通并获取数据（状态码 200）
            response.raise_for_status() 
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 定位帖子列表
            threads = soup.find_all("div", class_="thread-details") 
            
            for thread in threads:
                title_tag = thread.find("div", class_="thread-title")
                if title_tag:
                    title = title_tag.get_text(strip=True)
                    # 尝试获取内容摘要
                    snippet_tag = thread.find("div", class_="thread-content")
                    snippet = snippet_tag.get_text(strip=True) if snippet_tag else ""
                    
                    kaskus_data.append({
                        "platform": "Kaskus",
                        "text": f"{title} - {snippet}",
                        "author": "anonymous", 
                        "url": "https://www.kaskus.co.id" + title_tag.find("a")["href"] if title_tag.find("a") else "",
                        "created_at": ""
                    })
            
            # 礼貌性延时，防止请求过快被服务器临时封禁
            time.sleep(2)  
            
        except requests.exceptions.RequestException as e:
            print(f"抓取 Kaskus 第 {page} 页失败，网络报错: {e}")
            
    df = pd.DataFrame(kaskus_data)
    print(f"抓取结束。共成功获取 {len(df)} 条 Kaskus 数据。")
    return df

# --- 运行测试 ---
if __name__ == "__main__":
    # 测试抓取印尼网络赌博相关的黑话，抓取前 2 页
    test_keyword = "judi slot gacor"
    df_result = scrape_kaskus_search(test_keyword, pages=2)
    
    # 如果抓到了数据，将其保存为 CSV 文件
    if not df_result.empty:
        df_result.to_csv("kaskus_test_data.csv", index=False, encoding="utf-8-sig")
        print("数据已成功保存为 kaskus_test_data.csv")

正在抓取 Kaskus 第 1 页，关键词: judi slot gacor...
https://www.kaskus.co.id/search?q=judi slot gacor&page=1
抓取 Kaskus 第 1 页失败，网络报错: HTTPSConnectionPool(host='www.kaskus.co.id', port=443): Max retries exceeded with url: /search?q=judi%20slot%20gacor&page=1 (Caused by ProxyError('Unable to connect to proxy', NewConnectionError("HTTPSConnection(host='127.0.0.1', port=10808): Failed to establish a new connection: [Errno 111] Connection refused")))
正在抓取 Kaskus 第 2 页，关键词: judi slot gacor...
https://www.kaskus.co.id/search?q=judi slot gacor&page=2
抓取 Kaskus 第 2 页失败，网络报错: HTTPSConnectionPool(host='www.kaskus.co.id', port=443): Max retries exceeded with url: /search?q=judi%20slot%20gacor&page=2 (Caused by ProxyError('Unable to connect to proxy', NewConnectionError("HTTPSConnection(host='127.0.0.1', port=10808): Failed to establish a new connection: [Errno 111] Connection refused")))
抓取结束。共成功获取 0 条 Kaskus 数据。
